# Give Claude a Wallet: Agent Commerce with Azeth

This notebook shows how to give Claude economic agency — the ability to create accounts, discover services, make payments, and build on-chain reputation. We'll use [Azeth](https://azeth.ai) MCP tools to turn Claude into a participant in the machine economy.

**What you'll learn:**

1. How to define Azeth's MCP tools as Claude tool definitions
2. How Claude reasons about economic decisions (discover → evaluate → pay → rate)
3. The complete agent commerce loop: service discovery, reputation checking, x402 payment, and on-chain feedback

**What is Azeth?**

Azeth provides trust infrastructure for the machine economy:
- **Smart accounts** (ERC-4337) with guardian-enforced spending limits
- **Trust registry** (ERC-8004) for service discovery and on-chain identity
- **x402 payments** for HTTP-native machine-to-machine commerce
- **Reputation** — payment-gated, USD-weighted, on-chain feedback

In production, you'd use the [Azeth MCP Server](https://www.npmjs.com/package/@azeth/mcp-server) (`npx @azeth/mcp-server`) which provides 32 tools. Here we'll simulate the key tools to show Claude's decision-making.

## Setup

Install the Anthropic SDK:

In [ ]:
%pip install anthropic

In [ ]:
import anthropic
import json
from typing import Any

client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-20250514"

## Define Azeth tools

In production, you'd run `npx @azeth/mcp-server` and Claude would have all 32 tools available automatically via MCP. Here, we define the key tools as standard Claude tool definitions to show how the agent reasons about economic actions.

These tools mirror the real Azeth MCP server — same names, same parameters, same response format.

In [ ]:
azeth_tools = [
    {
        "name": "azeth_balance",
        "description": "Check the USDC and ETH balance of your smart account.",
        "input_schema": {
            "type": "object",
            "properties": {
                "address": {
                    "type": "string",
                    "description": 'Smart account address, or "me" for your own account.',
                }
            },
            "required": [],
        },
    },
    {
        "name": "azeth_discover_services",
        "description": "Search the ERC-8004 trust registry for services by capability. Returns services ranked by reputation score.",
        "input_schema": {
            "type": "object",
            "properties": {
                "capability": {
                    "type": "string",
                    "description": "Service capability to search for (e.g., 'weather-data', 'translation', 'image-generation').",
                },
                "min_reputation": {
                    "type": "number",
                    "description": "Minimum reputation score (0-100). Higher means more trusted.",
                },
                "limit": {
                    "type": "number",
                    "description": "Maximum number of results to return.",
                },
            },
            "required": ["capability"],
        },
    },
    {
        "name": "azeth_get_weighted_reputation",
        "description": "Get the USD-weighted reputation score for a service. Scores are payment-gated: only agents who paid can rate, weighted by payment volume.",
        "input_schema": {
            "type": "object",
            "properties": {
                "token_id": {
                    "type": "string",
                    "description": "Trust registry token ID of the service to check.",
                }
            },
            "required": ["token_id"],
        },
    },
    {
        "name": "azeth_pay",
        "description": "Pay for an x402-gated service. Sends USDC payment on-chain and returns the service response.",
        "input_schema": {
            "type": "object",
            "properties": {
                "url": {
                    "type": "string",
                    "description": "The URL of the x402-gated endpoint to pay for and access.",
                },
                "max_price": {
                    "type": "string",
                    "description": 'Maximum price willing to pay (e.g., "$0.01"). Rejects if service costs more.',
                },
            },
            "required": ["url"],
        },
    },
    {
        "name": "azeth_submit_opinion",
        "description": "Rate a service on-chain after using it. Only accounts that paid can rate. Reputation is payment-gated and USD-weighted.",
        "input_schema": {
            "type": "object",
            "properties": {
                "service_token_id": {
                    "type": "string",
                    "description": "Trust registry token ID of the service to rate.",
                },
                "success": {
                    "type": "boolean",
                    "description": "Whether the service call was successful.",
                },
                "quality_score": {
                    "type": "number",
                    "description": "Quality rating from 0-100.",
                },
            },
            "required": ["service_token_id", "success", "quality_score"],
        },
    },
]

print(f"Defined {len(azeth_tools)} Azeth tools: {[t['name'] for t in azeth_tools]}")

## Simulate Azeth infrastructure

In production, these tools hit real smart contracts on Base. For this notebook, we simulate the responses to focus on Claude's reasoning. The response format matches the real Azeth MCP server exactly.

In [ ]:
def simulate_tool_call(tool_name: str, tool_input: dict) -> dict[str, Any]:
    """Simulate Azeth MCP tool responses.

    In production, the Azeth MCP server handles these calls against real
    smart contracts on Base (L2) or Ethereum. The response format here
    matches the real server output exactly.
    """

    if tool_name == "azeth_balance":
        return {
            "success": True,
            "data": {
                "address": "0x742d35Cc6634C0532925a3b844Bc9e7595f2bD1e",
                "usdc": "12.50",
                "eth": "0.0041",
                "chain": "baseSepolia",
            },
        }

    elif tool_name == "azeth_discover_services":
        capability = tool_input.get("capability", "")
        if "weather" in capability:
            return {
                "success": True,
                "data": {
                    "services": [
                        {
                            "tokenId": "42",
                            "name": "WeatherOracle",
                            "entityType": "service",
                            "description": "Real-time weather data for AI agents",
                            "capabilities": ["weather-data"],
                            "endpoint": "https://weather.example.com",
                            "reputationScore": 87,
                            "totalInteractions": 1543,
                            "active": True,
                        },
                        {
                            "tokenId": "108",
                            "name": "ClimatePulse",
                            "entityType": "service",
                            "description": "Historical and forecast weather analytics",
                            "capabilities": ["weather-data", "climate-forecast"],
                            "endpoint": "https://climate.example.com",
                            "reputationScore": 72,
                            "totalInteractions": 389,
                            "active": True,
                        },
                    ],
                    "total": 2,
                },
            }
        return {"success": True, "data": {"services": [], "total": 0}}

    elif tool_name == "azeth_get_weighted_reputation":
        token_id = tool_input.get("token_id", "")
        scores = {
            "42": {
                "compositeScore": 87,
                "dimensions": {
                    "uptime": 99.2,
                    "responseTime": 92,
                    "successRate": 98.5,
                    "dataFreshness": 85,
                },
                "totalInteractions": 1543,
                "totalUsdVolume": "4521.30",
                "recentTrend": "stable",
            },
            "108": {
                "compositeScore": 72,
                "dimensions": {
                    "uptime": 94.1,
                    "responseTime": 68,
                    "successRate": 91.0,
                    "dataFreshness": 79,
                },
                "totalInteractions": 389,
                "totalUsdVolume": "892.10",
                "recentTrend": "improving",
            },
        }
        score = scores.get(token_id, scores["42"])
        return {"success": True, "data": score}

    elif tool_name == "azeth_pay":
        return {
            "success": True,
            "data": {
                "url": tool_input.get("url", ""),
                "amountPaid": "0.001",
                "currency": "USDC",
                "txHash": "0xabc123...def456",
                "response": {
                    "city": "London",
                    "temperature": 18,
                    "humidity": 65,
                    "conditions": "cloudy",
                    "timestamp": 1741276800,
                    "provider": "WeatherOracle",
                },
            },
        }

    elif tool_name == "azeth_submit_opinion":
        return {
            "success": True,
            "data": {
                "txHash": "0x789abc...123def",
                "serviceTokenId": tool_input.get("service_token_id", ""),
                "qualityScore": tool_input.get("quality_score", 0),
                "newCompositeScore": 88,
            },
        }

    return {"success": False, "error": {"code": "UNKNOWN_TOOL", "message": f"Unknown tool: {tool_name}"}}


print("Simulator ready. Tool calls will return realistic Azeth responses.")

## The agentic tool loop

This helper runs Claude in a loop: Claude receives a prompt, decides which tools to call, receives results, and continues until it has a final answer. This is the same pattern that Claude Code and Claude Desktop use with MCP servers.

In [ ]:
def run_agent(prompt: str, system: str = "") -> str:
    """Run Claude in an agentic tool-use loop with Azeth tools.

    Claude decides which tools to call, receives results, and continues
    until it produces a final text response. Each tool call is printed
    so you can observe the agent's reasoning.
    """
    messages = [{"role": "user", "content": prompt}]
    system_prompt = system or (
        "You are an AI agent with a crypto wallet powered by Azeth. "
        "You can discover services, check reputations, make payments, and rate providers. "
        "Always check your balance before making payments. "
        "Always check a service's reputation before paying. "
        "After receiving a service response, rate the provider honestly. "
        "Be concise in your reasoning."
    )

    while True:
        response = client.messages.create(
            model=MODEL,
            max_tokens=1024,
            system=system_prompt,
            tools=azeth_tools,
            messages=messages,
        )

        # Collect text and tool use blocks
        text_parts = []
        tool_calls = []
        for block in response.content:
            if block.type == "text":
                text_parts.append(block.text)
            elif block.type == "tool_use":
                tool_calls.append(block)

        # Print any reasoning
        if text_parts:
            print(f"\n{'='*60}")
            print(f"Claude: {''.join(text_parts)}")

        # If no tool calls, we're done
        if response.stop_reason == "end_turn" or not tool_calls:
            return "".join(text_parts)

        # Process each tool call
        tool_results = []
        for tool_call in tool_calls:
            print(f"\n  >> Tool: {tool_call.name}")
            print(f"     Input: {json.dumps(tool_call.input, indent=2)}")

            result = simulate_tool_call(tool_call.name, tool_call.input)
            print(f"     Result: {json.dumps(result, indent=2)[:300]}...")

            tool_results.append(
                {
                    "type": "tool_result",
                    "tool_use_id": tool_call.id,
                    "content": json.dumps(result),
                }
            )

        # Add assistant response and tool results to conversation
        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user", "content": tool_results})


print("Agent loop ready.")

## Scenario 1: Find and pay for weather data

Let's ask Claude to get weather data. Watch how it autonomously:
1. Checks its balance
2. Discovers services with the "weather-data" capability
3. Evaluates providers by reputation
4. Pays the best provider via x402
5. Rates the provider after receiving data

In [ ]:
result = run_agent(
    "I need current weather data for London. "
    "Find the best weather service, check its reputation, pay for the data, "
    "and rate the provider afterward."
)

## Scenario 2: Budget-conscious agent

Now let's see how Claude handles economic constraints. With a tight budget and a cost-conscious system prompt, the agent should still find data but be more careful about spending.

In [ ]:
result = run_agent(
    "Get me weather data for London. My budget is very limited — "
    "do not pay more than $0.01 for any single request. "
    "Compare available providers and pick the most cost-effective one with good reputation.",
    system=(
        "You are a budget-conscious AI agent. You have a crypto wallet powered by Azeth. "
        "Always check your balance first. Never overspend. "
        "Prefer providers with higher reputation scores — they waste less of your budget on retries. "
        "After using a service, always rate it to help other agents make better decisions."
    ),
)

## Using Azeth in production

The examples above simulate tool responses. In production, you have two options:

### Option 1: MCP Server (recommended for Claude Desktop / Claude Code)

Configure the Azeth MCP server and Claude gets all 32 tools automatically:

```json
{
  "mcpServers": {
    "azeth": {
      "command": "npx",
      "args": ["@azeth/mcp-server"],
      "env": {
        "AZETH_PRIVATE_KEY": "0x...",
        "PIMLICO_API_KEY": "your-key"
      }
    }
  }
}
```

### Option 2: TypeScript SDK (for custom applications)

```typescript
import { AzethKit } from '@azeth/sdk';

const agent = await AzethKit.create({
  privateKey: '0x...',
  chain: 'baseSepolia',
});

// Create smart account
await agent.createAccount({
  name: 'MyAgent',
  entityType: 'agent',
  description: 'An autonomous data analyst',
});

// Discover + pay + rate — all in one flow
const services = await agent.discoverServices({ capability: 'weather-data' });
const result = await agent.pay(`${services[0].endpoint}/api/weather/london`);
await agent.submitOpinion({
  serviceTokenId: services[0].tokenId,
  success: true,
  qualityScore: 90,
});
```

## Why this matters

The machine economy needs three primitives that traditional APIs don't provide:

1. **Trust** — How does an agent know which service to use? On-chain reputation scores, weighted by payment volume, provide a Sybil-resistant trust signal.

2. **Payment** — How does an agent pay for a service? x402 embeds payment into HTTP itself. No API keys, no billing accounts, no human approval.

3. **Identity** — How does an agent prove who it is? ERC-4337 smart accounts with ERC-8004 trust registry entries give every machine participant a verifiable on-chain identity.

Azeth composes these three primitives into a single SDK and MCP server, so any AI agent can participate in the machine economy.

## Resources

- [Azeth website](https://azeth.ai)
- [MCP Server on npm](https://www.npmjs.com/package/@azeth/mcp-server) — 32 tools for AI agents
- [TypeScript SDK](https://www.npmjs.com/package/@azeth/sdk)
- [Agent Starter Template](https://github.com/azeth-protocol/agent-starter) — fork and build in 5 minutes
- [Agent Commerce Demo](https://github.com/azeth-protocol/agent-commerce-demo) — two agents trading end-to-end
- [GitHub](https://github.com/azeth-protocol)